# 02metro: KORAIL metro rail (광역철도) workbooks

**What it does.** Reads KORAIL's yearly metro rail workbooks, which are laid out for printing (title rows, merged headings, subtotal rows between stations), and turns them into tidy tables. Every station total is checked against the workbook's own line subtotals.

- Boardings and alightings by line, station and month: 2019, 2024 and 2025.
- Transport (수송) by line and month, 2024 and 2025: boardings plus riders arriving from other lines.
- Gate taps by station and hour, every month of 2024 and 2025.

**Inputs:** `data/raw/korail_metro/` (workbooks C1 to C3).
**Outputs:** `data/clean/metro_station_month.csv`, `metro_line_month.csv`, `metro_hourly.csv`.

**Data credit:** Korea Railroad Corporation (KORAIL), metro rail transport statistics (info.korail.com, 광역철도 수송통계). Free to use.

## 0. Settings

In [1]:
from pathlib import Path
import re
import warnings
import openpyxl
import pandas as pd

RAW = Path("data/raw/korail_metro")  # <- folder with the workbooks
CLEAN = Path("data/clean")  # <- folder for the tidy tables

FILES_MONTHLY = {2024: RAW / "(2024년12월) 광역철도 수송(홈페이지).xlsx", 2025: RAW / "(2025년12월) 광역철도 수송(홈페이지).xlsx"}  # <- C2, C1
FILES_HOURLY = {2024: RAW / "(2024년12월) 광역철도 시간대별 승하차인원(홈페이지).xlsx", 2025: RAW / "(2025년12월) 광역철도 시간대별 승하차인원(홈페이지).xlsx"}  # <- C2, C1
FILE_2019 = RAW / "수송실적 201912_1(업로드).xlsx"  # <- C3
SHEET_2019 = "승하차"  # <- the 2019 sheet with stations ("2018년도 목표" is ignored)

CHECK_GYEONGBU_2025 = 240_764_709  # <- 2025 Gyeongbu line transport, from the workbook's own total

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
pd.set_option("display.width", 160)
print("pandas", pd.__version__)

pandas 3.1.0


## 1. Helpers

Rows come in pairs: a boardings row (승차) followed by an alightings row (하차). The alightings row sometimes has an empty or cut-off station name (for example 백 for 백양리), so each pair takes its line and station from the boardings row. A pair with a line but no station is that line's subtotal.

In [2]:
def sheet_rows(path, sheet):
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    rows = [list(r) for r in wb[sheet].iter_rows(values_only=True)]
    wb.close()
    return rows


def find_row(rows, text, start=0):
    for i in range(start, len(rows)):
        first = rows[i][0]
        if first is not None and text in str(first):
            return i
    return None


def clean_name(x):
    if x is None:
        return None
    return str(x).strip()


def read_pairs(rows, start, n_values, first_value_col=3):
    out = []
    i = start
    while i + 1 < len(rows):
        r = rows[i]
        if r[2] != "승차":
            i += 1
            continue
        nxt = rows[i + 1]
        if nxt[2] != "하차":
            i += 1
            continue
        line = clean_name(r[0])
        station = clean_name(r[1])
        board = r[first_value_col:first_value_col + n_values]
        alight = nxt[first_value_col:first_value_col + n_values]
        out.append((line, station, board, alight))
        i += 2
    return out

## 2. Boardings and alightings by station and month, 2024 and 2025

Sheet 승하차인원 has two blocks: "1) 노선별" (by line) and "2) 역별" (by station, with a subtotal pair after each line). Twelve month columns start in column D.

In [3]:
station_parts = []
line_parts = []
subtotal_parts = []

for year, path in FILES_MONTHLY.items():
    rows = sheet_rows(path, "승하차인원")
    b1 = find_row(rows, "1) 노선별")
    b2 = find_row(rows, "2) 역별")
    months = pd.date_range(f"{year}-01-01", periods=12, freq="MS")

    for line, station, board, alight in read_pairs(rows[:b2], b1, 12):
        for m, bo, al in zip(months, board, alight):
            line_parts.append((m, line, bo, al))

    current_line = None
    for line, station, board, alight in read_pairs(rows, b2, 12):
        current_line = line or current_line
        target = subtotal_parts if station is None else station_parts
        for m, bo, al in zip(months, board, alight):
            target.append((m, current_line, station, bo, al))

st_month = pd.DataFrame(station_parts, columns=["month", "line", "station", "board", "alight"])
sub_month = pd.DataFrame(subtotal_parts, columns=["month", "line", "station", "board", "alight"]).drop(columns="station")
line_month = pd.DataFrame(line_parts, columns=["month", "line", "board", "alight"])
print("station rows:", len(st_month), "| stations per year:", st_month.groupby(st_month["month"].dt.year)["station"].nunique().to_dict())
print("lines:", line_month["line"].unique().tolist())

station rows: 7752 | stations per year: {2024: 308, 2025: 308}
lines: ['광역철도 합계', 'ITX-청춘', '광역전철 합계', '경부선', '경인선', '경원선', '중앙선', '안산선', '과천선', '분당선', '일산선', '장항선', '경의선', '경춘선', '수인선', '경강선', '서해선', '동해선', '대경선']


In [4]:
by_line = st_month.groupby(["month", "line"])[["board", "alight"]].sum()
sub = sub_month.set_index(["month", "line"])
gap_sub = (by_line - sub.loc[by_line.index]).abs().max().max()
b1 = line_month.set_index(["month", "line"])
gap_b1 = (by_line - b1.loc[by_line.index]).abs().max().max()
print("largest gap, station sums against the line subtotals:", gap_sub)
print("largest gap, station sums against block 1 (by line):", gap_b1)

largest gap, station sums against the line subtotals: 0
largest gap, station sums against block 1 (by line): 0


## 3. Transport by line and month, 2024 and 2025

Sheet 수송: transport (수송량) = boardings on the line plus riders who boarded on another line and got off on this one. This is the measure KORAIL uses for line totals.

In [5]:
transport_parts = []
for year, path in FILES_MONTHLY.items():
    rows = sheet_rows(path, "수송")
    b1 = find_row(rows, "1) 노선별")
    b2 = find_row(rows, "2) 역별")
    months = pd.date_range(f"{year}-01-01", periods=12, freq="MS")
    for r in rows[b1 + 2:b2]:
        line = clean_name(r[0])
        if line is None:
            continue
        for m, v in zip(months, r[2:14]):
            transport_parts.append((m, line, v))

transport = pd.DataFrame(transport_parts, columns=["month", "line", "transport"])
line_month = line_month.merge(transport, on=["month", "line"], how="outer")
line_month["year"] = line_month["month"].dt.year

ty = line_month.groupby(["year", "line"])["transport"].sum().unstack(0) / 1e6
print("transport, millions:")
print(ty.round(1).sort_values(2025, ascending=False).to_string())
print()
print("Gyeongbu 2025 matches the workbook:", int(ty.loc["경부선", 2025] * 1e6 + 0.5) == CHECK_GYEONGBU_2025)

transport, millions:
year       2024    2025
line                   
광역철도 합계  1124.9  1123.4
광역전철 합계  1119.4  1118.2
경부선       242.0   240.8
분당선       206.7   204.5
경원선       154.1   154.5
경인선       141.2   139.3
일산선        59.4    60.3
과천선        55.9    56.5
경의선        57.3    54.1
안산선        53.5    52.7
중앙선        37.2    37.0
서해선        25.9    26.7
수인선        26.4    26.4
경춘선        18.7    19.2
동해선        17.3    17.2
경강선        16.7    16.9
장항선         7.0     7.1
ITX-청춘      5.4     5.2
대경선         0.3     5.1

Gyeongbu 2025 matches the workbook: True


## 4. Boardings and alightings by station and month, 2019

The 2019 workbook has no line column. Each line's station pairs are followed by a total pair named "<line> 합계" (or "<line>합계"), so stations are given the line of the next total row. ITX-청춘 has its own block at the end. Averages (일평균) and the network totals are skipped.

In [6]:
rows = sheet_rows(FILE_2019, SHEET_2019)
months19 = pd.date_range("2019-01-01", periods=12, freq="MS")
pairs = read_pairs(rows, 4, 12)

parts19 = []
totals19 = {}
pending = []
for line, station, board, alight in pairs:
    name = station or ""
    if "일평균" in name:
        continue
    if "합계" in name:
        label = name.replace("합계", "").strip()
        if label in ("광역철도", "일반전철"):
            totals19[label] = sum(board)
            continue
        totals19[label] = sum(board)
        for s, bo_list, al_list in pending:
            for m, bo, al in zip(months19, bo_list, al_list):
                parts19.append((m, label, s, bo, al))
        pending = []
        continue
    pending.append((name, board, alight))

st19 = pd.DataFrame(parts19, columns=["month", "line", "station", "board", "alight"])
print("lines:", st19["line"].unique().tolist())
print("stations:", st19["station"].nunique(), "| left without a line:", len(pending))

sums19 = st19.groupby("line")["board"].sum()
gaps = {k: int(sums19.get(k, 0) - v) for k, v in totals19.items() if k not in ("광역철도", "일반전철")}
print("station sums minus line totals (boardings):", gaps)
print("network total boardings 2019:", totals19["광역철도"], "| station sum:", int(st19["board"].sum()))

lines: ['경부선', '경인선', '경원선', '중앙선', '안산선', '과천선', '분당선', '일산선', '장항선', '경의선', '경춘선', '수인선', '경강선', '동해선', 'ITX-청춘']
stations: 266 | left without a line: 0
station sums minus line totals (boardings): {'경부선': 0, '경인선': 0, '경원선': 0, '중앙선': 0, '안산선': 0, '과천선': 0, '분당선': 0, '일산선': 0, '장항선': 0, '경의선': 0, '경춘선': 0, '수인선': 0, '경강선': 0, '동해선': 0, 'ITX-청춘': 0}
network total boardings 2019: 814475374 | station sum: 814475374


In [7]:
lines19 = st19.groupby(["month", "line"])[["board", "alight"]].sum().reset_index()
lines19["year"] = 2019
line_month = pd.concat([lines19, line_month], ignore_index=True)
st_all = pd.concat([st19, st_month], ignore_index=True)
st_all["year"] = st_all["month"].dt.year

net = st_all.groupby("year")["board"].sum() / 1e6
print("boardings, millions:", net.round(1).to_dict())
print("2025 against 2019:", round((net[2025] / net[2019] - 1) * 100, 1), "%")

boardings, millions: {2019: 814.5, 2024: 783.6, 2025: 782.8}
2025 against 2019: -3.9 %


## 5. Gate taps by station and hour, 2024 and 2025

One sheet per month; the sheet order differs between the two years, so the month is read from each sheet's tab name, for example "(6월)시간대별 승하차". The titles inside the sheets are not used for this: the June 2024 sheet's title says "1900년 1월", a slip in KORAIL's workbook. Any title that disagrees with its tab is listed. Column labels run from "06시 이전" to "24시 이후". They are read as the hour in which the tap happened: **"08시" = 08:00 to 08:59**, "06시 이전" = before 07:00 and "24시 이후" = after midnight. KORAIL does not state this; it fits the data (morning arrivals peak in "08시", evening boardings in "18시").

These counts are **gate taps only**. Across the network they come to 96 to 100% of the monthly boardings, but at single stations shared with other operators they can be far lower, so they are used for the shape of the day, not for station totals.

In [8]:
HOUR_LABELS = None
hourly_parts = []
title_slips = []
for year, path in FILES_HOURLY.items():
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    for ws in wb.worksheets:
        rows = [list(r) for r in ws.iter_rows(values_only=True)]
        tab_month = int(re.search(r"(\d{1,2})월", ws.title).group(1))
        month = pd.Timestamp(year, tab_month, 1)
        title = str(rows[1][0])
        if f"{year}년 {tab_month}월" not in title.replace("년 0", "년 "):
            title_slips.append((ws.title, title.strip()))
        b2 = find_row(rows, "2) 역별")
        header = rows[b2 + 1]
        HOUR_LABELS = [clean_name(h) for h in header[3:22]]
        current_line = None
        for line, station, board, alight in read_pairs(rows, b2, 20):
            current_line = line or current_line
            if station is None:
                continue
            for label, bo, al in zip(HOUR_LABELS + ["합계"], board, alight):
                hourly_parts.append((month, current_line, station, label, bo, al))
    wb.close()

hourly = pd.DataFrame(hourly_parts, columns=["month", "line", "station", "hour_label", "board", "alight"])
print("hour labels:", HOUR_LABELS)
print("months:", hourly["month"].nunique(), "|", hourly["month"].min().date(), "to", hourly["month"].max().date())
print("sheet titles that disagree with their tab:", title_slips)

hour labels: ['06시 이전', '07시', '08시', '09시', '10시', '11시', '12시', '13시', '14시', '15시', '16시', '17시', '18시', '19시', '20시', '21시', '22시', '23시', '24시 이후']
months: 24 | 2024-01-01 to 2025-12-01
sheet titles that disagree with their tab: [('(6월)시간대별 승하차', '◈ 1900년 1월 시간대별 승하차인원')]


In [9]:
totals = hourly[hourly["hour_label"] == "합계"].set_index(["month", "line", "station"])[["board", "alight"]]
hourly = hourly[hourly["hour_label"] != "합계"].copy()
hour_start = {label: i + 6 for i, label in enumerate(HOUR_LABELS)}
hourly["hour"] = hourly["hour_label"].map(hour_start)

row_sums = hourly.groupby(["month", "line", "station"])[["board", "alight"]].sum()
print("largest gap, hours against each row's own total:", (row_sums - totals).abs().max().max())

taps = hourly.groupby("month")["board"].sum()
monthly_board = st_month.groupby("month")["board"].sum()
ratio = (taps / monthly_board).dropna()
print("gate taps as a share of monthly boardings:", round(ratio.min(), 3), "to", round(ratio.max(), 3))

largest gap, hours against each row's own total: 0
gate taps as a share of monthly boardings: 0.964 to 1.002


In [10]:
profile = hourly.groupby([hourly["month"].dt.year, "hour"])[["board", "alight"]].sum()
profile = profile / profile.groupby(level=0).transform("sum") * 100
print("share of the day's boardings and alightings by hour, percent:")
print(profile.unstack(0).round(1).to_string())

share of the day's boardings and alightings by hour, percent:
      board      alight      
month  2024 2025   2024  2025
hour                         
6       6.7  6.6    2.7   2.6
7       9.7  9.6    4.7   4.7
8       9.4  9.3    8.7   8.7
9       6.1  6.1    5.8   5.8
10      4.9  5.0    4.4   4.4
11      4.9  4.9    4.3   4.3
12      5.1  5.2    4.6   4.7
13      5.1  5.1    4.9   4.9
14      4.9  5.0    4.8   4.9
15      5.6  5.7    5.3   5.4
16      6.5  6.5    6.3   6.4
17      7.8  7.8    8.0   8.1
18      8.3  8.2   10.5  10.4
19      4.5  4.5    8.3   8.1
20      3.8  3.8    5.2   5.2
21      3.2  3.2    4.5   4.5
22      2.4  2.4    4.0   3.9
23      1.0  1.0    2.6   2.5
24      0.1  0.1    0.7   0.6


## 6. Save

In [11]:
st_all = st_all[["year", "month", "line", "station", "board", "alight"]]
line_month = line_month[["year", "month", "line", "board", "alight", "transport"]].sort_values(["line", "month"])
hourly = hourly[["month", "line", "station", "hour", "hour_label", "board", "alight"]]

st_all.to_csv(CLEAN / "metro_station_month.csv", index=False, encoding="utf-8-sig")
line_month.to_csv(CLEAN / "metro_line_month.csv", index=False, encoding="utf-8-sig")
hourly.to_csv(CLEAN / "metro_hourly.csv", index=False, encoding="utf-8-sig")
for f in sorted(CLEAN.glob("metro_*.csv")):
    print(f.name, f.stat().st_size // 1024, "KB")

metro_hourly.csv 6876 KB
metro_line_month.csv 33 KB
metro_station_month.csv 552 KB


___
**END**